In [ ]:
import json
from collections import defaultdict


from pipelines.shared.string_utils import split_camel_case
# Setup

%load_ext autoreload
%autoreload 2

import os

import torch
from umap import UMAP
from torch.nn.functional import normalize
from dotenv import load_dotenv
from pandas import read_json
from sklearn.cluster import HDBSCAN
from transformers import AutoModel, AutoTokenizer

from pipelines.multi_service.mean_pooling import mean_pooling

load_dotenv()

print(f"HuggingFace cache directory is {os.environ.get('HF_HOME')}")


def should_be_skipped(class_name: str):
    return (
        class_name.endswith("Test")
        or class_name.endswith("Tests")
        or class_name.endswith("IT")
    )

In [ ]:
import pandas as pd

# MODEL = "google-bert/bert-base-uncased"
# MODEL = "google-bert/bert-base-cased"
GENERIC_WORDS = [
    "get",
    "set",
    "find by",
    "exists by",
    "exists",
    "update",
    "delete",
    "delete by",
    "create",
    "get all",
    "create",
    "insert",
    "insert into",
    "find all",
    "find all by",
]
MODEL = "microsoft/codebert-base"

codeBertTokenizer = AutoTokenizer.from_pretrained(MODEL)
codeBertModel = AutoModel.from_pretrained(MODEL)

# bertTokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
# bertModel = AutoModel.from_pretrained("google-bert/bert-base-uncased")

# input_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service.jsonl"
# output_file = "/media/martin/BigData/datasets/pa165-git/01/multi-service-result.jsonl"

input_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/multi-service-labeled.jsonl"
)
output_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service-final-multiply_signature-remove-words_yes.jsonl"

# input_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service.json"
# output_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service-result-codebert-combined.jsonl"

data_frame = read_json(input_file, lines=True)

# origin_property = "methodSignature"
# evaluation_property = "methodSignatures"


def extract_property(methods: list[dict[str, str]], property: str):
    return list(map(lambda m: m[property], methods))


def remove_words(text: str):
    for word in GENERIC_WORDS:
        if text.startswith(word):
            text = text.removeprefix(word).strip()

    return text


def evaluate(inputs: list[dict[str, str]]):
    mean_pooled_signature = get_embeddings(extract_property(inputs, "signature"))
    method_names = extract_property(inputs, "name")
    method_name_split = list(
        map(lambda name: remove_words(" ".join(split_camel_case(name))), method_names)
    )
    mean_names = get_embeddings(method_name_split)

    signature_normalized = normalize(mean_pooled_signature, p=2, dim=1)
    name_normalized = normalize(mean_names, p=2, dim=1)

    combined = torch.cat((1.3 * signature_normalized, name_normalized), dim=-1)
    mean_pooled_numpy = combined.cpu().numpy()

    # pca = PCA(n_components=0.95, random_state=42)
    # pca_reduced = pca.fit_transform(mean_pooled_numpy)

    umap = UMAP(
        n_neighbors=5,
        min_dist=0.0,
        n_components=15,
        metric="cosine",
        random_state=42,
        init="random",
    )
    umap_reduced = umap.fit_transform(mean_pooled_numpy)
    # gap_df = calculate_gap_statistic(
    #     pca_reduced, n_refs=5, max_k=math.ceil(math.sqrt(len(inputs)))
    # )

    clustering_alg = HDBSCAN(
        min_samples=1, min_cluster_size=3, metric="euclidean", allow_single_cluster=True
    )
    labels = clustering_alg.fit_predict(umap_reduced)

    unique_labels = set(labels)
    number_of_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    noise_count = list(labels).count(-1)
    noise_ratio = noise_count / len(labels)
    # return (get_optimal_k_programmatically(gap_df)["optimal_k"], pca_reduced)

    if all(x == -1 for x in labels):
        number_of_clusters = -1

    return (number_of_clusters, noise_ratio, umap_reduced, labels)


def get_embeddings(inputs, tokenizer=codeBertTokenizer, model=codeBertModel):
    tokenized_inputs = tokenizer(
        inputs, return_tensors="pt", padding=True, truncation=True, max_length=256
    )
    with torch.no_grad():
        output = model(**tokenized_inputs)
    attention_mask = tokenized_inputs["attention_mask"]
    mean_pooled = mean_pooling(output, attention_mask)
    return mean_pooled


data_frame["optimal_k"] = pd.Series()
data_frame["noise_ratio"] = pd.Series()
data_frame["cluster"] = pd.Series()
results = []
for index, row in data_frame.iterrows():
    class_name = row["fullyQualifiedName"]
    methods = row["methods"]

    if len(methods) < 10:
        print(f"Skipped small class {class_name}, number of methods {len(methods)}")
        continue

    if should_be_skipped(class_name):
        print(f"Skipped Test class {class_name}, number of methods {len(methods)}")
        continue

    optimal_k, noise_ratio, pca_reduced, labels = evaluate(methods)

    # kmeans = KMeans(n_clusters=optimal_k, random_state=42)
    # cluster = kmeans.fit_predict(pca_reduced)
    data_frame.loc[index, "optimal_k"] = optimal_k
    data_frame.loc[index, "noise_ratio"] = noise_ratio
    # data_frame.at[index, "cluster"] = cluster.tolist()
    data_frame.at[index, "cluster"] = labels

    results.append(f"{class_name}: {optimal_k}, {labels}")

for result in results:
    print(result)

data_frame.to_json(output_file, lines=True, orient="records")

In [ ]:
input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service-final-multiply_signature-remove-words_yes.jsonl"
# input_file = "/media/martin/BigData/datasets/github-projects/cas/multi-service-result-codebert-combined.jsonl"

df = pd.read_json(input_file, lines=True)
one_cluster_lines = df.query("optimal_k == 1 and noise_ratio <= 0.3")
more_cluster_lines = df.query("optimal_k > 1 or noise_ratio > 0.3")


def count_rows_and_print(df: pd.DataFrame, query: str, stat_type: str):
    df_part = df.query(query)
    count = len(df_part)
    print(f"{stat_type}: {count}")
    for _, record in df_part.iterrows():
        print(record["fullyQualifiedName"])
        # print(json.dumps(record["methods"], indent=2))
    print("---")
    return count


false_negatives = count_rows_and_print(
    one_cluster_lines, "label == 1", "False negatives"
)
true_negatives = count_rows_and_print(one_cluster_lines, "label == 0", "True negatives")
false_positives = count_rows_and_print(
    more_cluster_lines, "label == 0", "False positives"
)
true_positives = count_rows_and_print(
    more_cluster_lines, "label == 1", "True positives"
)

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)
accuracy = (true_positives + true_negatives) / (
    true_positives + false_positives + false_negatives + true_negatives
)
f1_score = 2 * ((precision * recall) / (precision + recall))

print(f"""
Stats
Precision: {precision}
Recall: {recall}
Accuracy: {accuracy}
F1 score: {f1_score}
-------------
""")

class_clusters = defaultdict(dict)

for _, record in more_cluster_lines.iterrows():
    k = record["optimal_k"]
    clusters = record["cluster"]
    methods = record["methods"]
    class_name = record["fullyQualifiedName"]
    cohesion = record["lcom4"]
    noise_ratio = record["noise_ratio"]

    method_clusters = {}
    for index, cluster in enumerate(clusters):
        cluster_number_str = str(cluster)
        method = methods[index]
        if cluster_number_str in method_clusters:
            method_clusters[cluster_number_str].append(method)
        else:
            method_clusters[cluster_number_str] = [method]

    class_clusters[class_name]["methods"] = method_clusters
    class_clusters[class_name]["lcom4"] = cohesion
    class_clusters[class_name]["noise_ratio"] = noise_ratio

for className, metadata in class_clusters.items():
    print(f"--- Class {className}, LCOM4: {metadata['lcom4']} ---")
    for cluster, methods in metadata["methods"].items():
        print(f"Cluster {cluster}: {json.dumps(methods, indent=2)}\n")
    print()

more_cluster_lines

In [ ]:
from pipelines.shared import input_until_integer

# Labeling script

input_file = "/media/martin/BigData/datasets/github-projects/klaw/multi-service.jsonl"
output_file = (
    "/media/martin/BigData/datasets/github-projects/klaw/multi-service-labeled.jsonl"
)

df = read_json(input_file, lines=True)

df["label"] = pd.Series()

for index, row in df.iterrows():
    name = row["fullyQualifiedName"]
    methods = row["methods"]
    assigned_label = row["label"]
    lcom4 = row["lcom4"]

    label = 0
    if not should_be_skipped(name) and len(methods) > 10:
        print("-" * 10)
        print(f"Class {name} (LCOM4 {lcom4})")
        print("Methods: ")
        for method in methods:
            print(method)

        label = input_until_integer("Enter label: ")

    df.loc[index, "label"] = label
    df.to_json(output_file, lines=True, orient="records")

# Results

## Remove words: yes, names multiply: 1.3

```
False negatives: 1
io.aiven.klaw.service.UtilControllerService
---
True negatives: 10
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.ClusterApiService
---
True positives: 15
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.625
Recall: 0.9375
Accuracy: 0.7142857142857143
F1 score: 0.75
```

## Remove words: yes, names multiply: no

```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 13
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 6
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.7
Recall: 0.875
Accuracy: 0.7714285714285715
F1 score: 0.7777777777777777
-------------
```

## Remove words: no, names multiply: no

```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 10
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 9
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.6086956521739131
Recall: 0.875
Accuracy: 0.6857142857142857
F1 score: 0.717948717948718
-------------
```

## Remove words: no, names multiply: 1.3
```
False negatives: 2
io.aiven.klaw.clusterapi.controller.ClusterApiController
io.aiven.klaw.service.UtilControllerService
---
True negatives: 9
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.AnalyticsControllerService
io.aiven.klaw.service.MailUtils
---
False positives: 10
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.ClusterApiService
---
True positives: 14
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.5833333333333334
Recall: 0.875
Accuracy: 0.6571428571428571
F1 score: 0.7000000000000001
```

## Remove words: yes, signature multiply: 1.3

```
False negatives: 1
io.aiven.klaw.clusterapi.controller.ClusterApiController
---
True negatives: 10
io.aiven.klaw.clusterapi.services.ConfluentCloudApiService
io.aiven.klaw.MockMethods
io.aiven.klaw.controller.SchemaRegistryController
io.aiven.klaw.repository.MessageSchemaRepo
io.aiven.klaw.repository.KwKafkaConnectorRepo
io.aiven.klaw.controller.TopicController
io.aiven.klaw.repository.SchemaRequestRepo
io.aiven.klaw.controller.KafkaConnectController
io.aiven.klaw.service.MailUtils
io.aiven.klaw.service.ClusterApiService
---
False positives: 9
io.aiven.klaw.repository.AclRequestsRepo
io.aiven.klaw.repository.AclRepo
io.aiven.klaw.repository.ActivityLogRepo
io.aiven.klaw.controller.AclController
io.aiven.klaw.repository.KwKafkaConnectorRequestsRepo
io.aiven.klaw.repository.TopicRequestsRepo
io.aiven.klaw.repository.RegisterInfoRepo
io.aiven.klaw.repository.TopicRepo
io.aiven.klaw.service.AnalyticsControllerService
---
True positives: 15
io.aiven.klaw.clusterapi.utils.ClusterApiUtils
io.aiven.klaw.UtilMethods
io.aiven.klaw.helpers.HandleDbRequests
io.aiven.klaw.controller.TemplateMapController
io.aiven.klaw.controller.UsersTeamsController
io.aiven.klaw.controller.EnvsClustersTenantsController
io.aiven.klaw.helpers.db.rdbms.SelectDataJdbc
io.aiven.klaw.helpers.db.rdbms.HandleDbRequestsJdbc
io.aiven.klaw.helpers.db.rdbms.InsertDataJdbc
io.aiven.klaw.helpers.db.rdbms.DeleteDataJdbc
io.aiven.klaw.service.ServerConfigService
io.aiven.klaw.service.UsersTeamsControllerService
io.aiven.klaw.service.UtilControllerService
io.aiven.klaw.service.CommonUtilsService
io.aiven.klaw.clusterapi.UtilMethods
---

Stats
Precision: 0.625
Recall: 0.9375
Accuracy: 0.7142857142857143
F1 score: 0.75
```
